In [1]:
import spacy
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import gensim.downloader as api
from datasets import load_dataset
from sklearn.metrics import classification_report
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.utils import compute_class_weight
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from task6.torch_utils.model_callbacks import EarlyStopping, ModelCheckpoint
from task6.utils.balance_dataset import augment_minority_classes
from task6.utils.prepare_data import prepare_data
from task6.utils.prepare_rnn_data import prepare_rnn_data

C:\Projects\BUAS\Y2\block-a\fae2-nlpr-group-group-9-1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Szymon\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
import torch
import torchvision

print("Torch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


Torch version: 2.8.0+cu129
Torchvision version: 0.23.0+cu129
CUDA available: True
CUDA device: NVIDIA GeForce RTX 5080


In [3]:
dataset = load_dataset("google-research-datasets/go_emotions")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})

In [4]:
df = pd.DataFrame(dataset["train"])
df.head()

,text,labels,id
0,My favourite food is anything I didn't have to...,[27],eebbqej
1,"Now if he does off himself, everyone will thin...",[27],ed00q6i
2,WHY THE FUCK IS BAYLESS ISOING,[2],eezlygj
3,To make her feel threatened,[14],ed7ypvh
4,Dirty Southern Wankers,[3],ed0bdzj


In [5]:
df, emotions = prepare_data(df, "text", "labels")
print(emotions)
df.head()

Detected dataset type: goemotions
Starting batch preprocessing...
✓ Text cleaning completed
✓ All NLP processing completed
['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']


,text,id,ekman_emotion,tokenized_text,lemmatized_text
0,My favourite food is anything I didn't have to...,eebbqej,6,"[my, favourite, food, is, anything, i, did, n'...","[my, favourite, food, be, anything, I, do, not..."
1,"Now if he does off himself, everyone will thin...",ed00q6i,6,"[now, if, he, does, off, himself, ,, everyone,...","[now, if, he, do, off, himself, ,, everyone, w..."
2,WHY THE FUCK IS BAYLESS ISOING,eezlygj,0,"[why, the, fuck, is, bayless, isoing]","[why, the, fuck, be, bayless, isoe]"
3,To make her feel threatened,ed7ypvh,2,"[to, make, her, feel, threatened]","[to, make, she, feel, threaten]"
4,Dirty Southern Wankers,ed0bdzj,0,"[dirty, southern, wankers]","[dirty, southern, wanker]"


In [6]:
df.ekman_emotion.value_counts()

ekman_emotion
3    16920
6    12823
0     5579
5     4160
4     2599
2      691
1      638
Name: count, dtype: int64

In [7]:
df_test = pd.read_csv("../../data/transcript_spell_checked.csv")
df_test, emotions = prepare_data(df_test, "Translation", "emotion_final")

Detected dataset type: transcript
Starting batch preprocessing...
✓ Text cleaning completed
✓ All NLP processing completed


In [8]:
target_samples = 2500

classes_to_augment = df["ekman_emotion"].value_counts()[df["ekman_emotion"].value_counts() < target_samples].index.tolist()
print(f"Classes to augment: {classes_to_augment}")

Classes to augment: [2, 1]


In [9]:
df_augmented = augment_minority_classes(df, target_samples=target_samples, classes_to_augment=classes_to_augment)
df_augmented = pd.concat([df, df_augmented]).reset_index(drop=True)
print(df_augmented["ekman_emotion"].value_counts())
print(f"Original size: {len(df)}, Augmented size: {len(df_augmented)}")
df = df_augmented

Augmenting emotion 2: 691 → 2500
Augmenting emotion 1: 638 → 2500
Augmented Sample 1: I ’m so nervous ! thence
Augmented Sample 2: omg creepy !
Augmented Sample 3: I be kinda afraid to ask but do anyone reply ?
ekman_emotion
3    16920
6    12823
0     5579
5     4160
4     2599
2     2500
1     2500
Name: count, dtype: int64
Original size: 43410, Augmented size: 47081


In [10]:
df["lemmatized_text"] = df["lemmatized_text"].apply(lambda tokens: " ".join(tokens))
df_test["lemmatized_text"] = df_test["lemmatized_text"].apply(lambda tokens: " ".join(tokens))

In [11]:
word2vec_model = api.load("word2vec-google-news-300")

In [12]:
nlp = spacy.load("en_core_web_sm")

In [13]:
def get_pos_tags(text, nlp):
    doc = nlp(text)
    pos_tags = [token.pos_ for token in doc]
    return pos_tags

In [14]:
df["pos_tags"] = df["text"].apply(lambda x: get_pos_tags(x, nlp))
df_test["pos_tags"] = df_test["Sentence"].apply(lambda x: get_pos_tags(x, nlp))

In [15]:
df.head()

,text,id,ekman_emotion,tokenized_text,lemmatized_text,pos_tags
0,My favourite food is anything I didn't have to...,eebbqej,6,"[my, favourite, food, is, anything, i, did, n'...",my favourite food be anything I do not have to...,"[PRON, ADJ, NOUN, AUX, PRON, PRON, AUX, PART, ..."
1,"Now if he does off himself, everyone will thin...",ed00q6i,6,"[now, if, he, does, off, himself, ,, everyone,...","now if he do off himself , everyone will think...","[INTJ, SCONJ, PRON, VERB, ADP, PRON, PUNCT, PR..."
2,WHY THE FUCK IS BAYLESS ISOING,eezlygj,0,"[why, the, fuck, is, bayless, isoing]",why the fuck be bayless isoe,"[SCONJ, DET, NOUN, AUX, PROPN, PROPN]"
3,To make her feel threatened,ed7ypvh,2,"[to, make, her, feel, threatened]",to make she feel threaten,"[PART, VERB, PRON, VERB, VERB]"
4,Dirty Southern Wankers,ed0bdzj,0,"[dirty, southern, wankers]",dirty southern wanker,"[ADJ, PROPN, PROPN]"


In [16]:
X_train, y_train = prepare_rnn_data(df, "lemmatized_text", word2vec_model, "ekman_emotion", additional_columns = ["pos_tags"])
X_test, y_test = prepare_rnn_data(df_test, "lemmatized_text", word2vec_model, "ekman_emotion", additional_columns = ["pos_tags"])
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Preparing RNN data from 47081 samples...
Embedding dimension: 300
Max sequence length: 50


Converting to embeddings:  23%|██▎       | 10946/47081 [00:00<00:02, 16047.48it/s]<string>:1: SyntaxWarning: invalid decimal literal
<string>:1: SyntaxWarning: invalid decimal literal
Converting to embeddings: 100%|██████████| 47081/47081 [00:02<00:00, 15858.36it/s]


Processing additional column: pos_tags
  Found 18 unique values in pos_tags
✓ Final shape: (47081, 50, 301)
✓ OOV rate: 22.15% (163612/738538)
✓ Data type: float64
Preparing RNN data from 914 samples...
Embedding dimension: 300
Max sequence length: 50


Converting to embeddings: 100%|██████████| 914/914 [00:00<00:00, 11624.88it/s]

Processing additional column: pos_tags
  Found 15 unique values in pos_tags


✓ Final shape: (914, 50, 301)
✓ OOV rate: 25.58% (2091/8173)
✓ Data type: float64
Train shape: (47081, 50, 301), Test shape: (914, 50, 301)


In [17]:
class EmotionDataset(Dataset):
    """PyTorch Dataset for emotion classification"""

    def __init__(self, sequences, labels):
        self.sequences = torch.FloatTensor(sequences)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

In [18]:
class BiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3):
            super().__init__()

            # Input normalization
            self.input_norm = nn.LayerNorm(input_size)

            # BiLSTM with proper initialization
            self.lstm = nn.LSTM(
                input_size,
                hidden_size,
                num_layers,
                batch_first=True,
                dropout=dropout if num_layers > 1 else 0,
                bidirectional=True
            )

            # Attention mechanism (helps with long sequences)
            self.attention = nn.MultiheadAttention(
                embed_dim=hidden_size * 2,
                num_heads=8,
                dropout=dropout,
                batch_first=True
            )

            # Classification layers
            self.dropout = nn.Dropout(dropout)
            self.batch_norm = nn.BatchNorm1d(hidden_size * 2)

            # Simpler classifier
            self.classifier = nn.Sequential(
                nn.Linear(hidden_size * 2, hidden_size),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_size, num_classes)
            )

            # Initialize weights properly
            self._init_weights()

    def _init_weights(self):
            """Proper weight initialization"""
            for name, param in self.named_parameters():
                if 'weight' in name:
                    if 'lstm' in name:
                        nn.init.xavier_uniform_(param)
                    elif 'linear' in name or 'fc' in name:
                        nn.init.xavier_uniform_(param)
                elif 'bias' in name:
                    nn.init.constant_(param, 0)

    def forward(self, x):
            # Input normalization
            x = self.input_norm(x)

            # BiLSTM
            lstm_out, (hidden, cell) = self.lstm(x)
            lstm_out = self.dropout(lstm_out)

            # Attention mechanism (instead of just taking last output)
            attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)

            # Global max pooling across sequence dimension
            pooled = torch.max(attn_out, dim=1)[0]  # Take max across sequence

            # Batch normalization
            pooled = self.batch_norm(pooled)

            # Classification
            output = self.classifier(pooled)

            return output

In [19]:
# train / validation split
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train, shuffle=True)

X_train = np.array(X_train)
y_train = np.array(y_train)
X_val = np.array(X_val)
y_val = np.array(y_val)

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [21]:
num_classes = len(emotions)
epochs = 250
patience = 10
lr_patience = 10
model_save_path = "../../models/BiLSTM/best_model.pth"
num_classes

7

In [22]:
def create_better_training_setup():
    """
    Create better training configuration
    """

    training_config = {
        # Model architecture
        'input_size': 301,
        'hidden_size': 128,  # Smaller hidden size
        'num_layers': 1,     # Fewer layers
        'dropout': 0.2,      # Lower dropout

        # Training parameters
        'learning_rate': 0.001,   # Higher learning rate
        'weight_decay': 1e-5,     # Lower weight decay
        'batch_size': 64,         # Smaller batch size
        'patience': 15,           # More patience
        'lr_patience': 5,         # Reduce LR sooner

        # Class weights (more aggressive for minorities)
        'custom_class_weights': compute_class_weight(
            class_weight='balanced',
            classes=np.arange(num_classes),
            y=y_train
        ).tolist()
    }

    return training_config

In [23]:
def train_bilstm(X_train, X_val, X_test, y_train, y_val, y_test, emotions, device, save_path=model_save_path):
    """
    Train BiLSTM with all fixes applied
    """

    print("TRAINING FIXED BILSTM MODEL")
    print("="*35)

    # Get fixed training configuration
    config = create_better_training_setup()

    # Create datasets with better batch size
    train_dataset = EmotionDataset(X_train, y_train)
    val_dataset = EmotionDataset(X_val, y_val)
    test_dataset = EmotionDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)

    # Create fixed model
    model = BiLSTM(
        input_size=config['input_size'],
        hidden_size=config['hidden_size'],
        num_layers=config['num_layers'],
        num_classes=len(emotions),
        dropout=config['dropout']
    ).to(device)

    print(f"Model architecture:")
    print(f"  Hidden size: {config['hidden_size']}")
    print(f"  Num layers: {config['num_layers']}")
    print(f"  Dropout: {config['dropout']}")
    print(f"  Batch size: {config['batch_size']}")

    # Better class weights
    class_weights = torch.FloatTensor([
        config['custom_class_weights'][i] for i in range(len(emotions))
    ]).to(device)

    print(f"Class weights: {class_weights.cpu().numpy()}")

    # Better optimizer and loss
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)  # Add label smoothing
    optimizer = optim.AdamW(  # Use AdamW instead of Adam
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )

    # Better scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )

    print(f"Learning rate: {config['learning_rate']}")
    print(f"Weight decay: {config['weight_decay']}")
    print(f"Using label smoothing: 0.1")

    # Training loop with better monitoring
    best_val_acc = 0
    patience_counter = 0

    for epoch in range(500):  # Fewer max epochs, focus on quality
        # Training
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for sequences, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}'):
            sequences, labels = sequences.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(sequences)
            loss = criterion(outputs, labels)

            # Check for NaN
            if torch.isnan(loss):
                print("⚠️  NaN loss detected! Stopping training.")
                break

            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        # Validation
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        all_val_preds = []
        all_val_labels = []

        with torch.no_grad():
            for sequences, labels in val_loader:
                sequences, labels = sequences.to(device), labels.to(device)
                outputs = model(sequences)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

                all_val_preds.extend(predicted.cpu().numpy())
                all_val_labels.extend(labels.cpu().numpy())

        # Calculate metrics
        epoch_train_loss = train_loss / len(train_loader)
        epoch_train_acc = 100 * train_correct / train_total
        epoch_val_loss = val_loss / len(val_loader)
        epoch_val_acc = 100 * val_correct / val_total
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Epoch {epoch+1:2d}: Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:5.2f}%, "
              f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:5.2f}%, LR: {current_lr:.2e}")

        # Check prediction diversity
        unique_preds = len(set(all_val_preds))
        print(f"           Predicting {unique_preds}/7 different emotions")

        # Early stopping with improvement
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            patience_counter = 0
            # Save best model
            torch.save(model.state_dict(), save_path)
            print(f"Saved best model with Val Acc: {best_val_acc:.2f}% on epoch {epoch+1}")
        else:
            patience_counter += 1

        if patience_counter >= config['patience']:
            print(f"Early stopping at epoch {epoch+1}")
            break

    # Load best model and evaluate
    model.load_state_dict(torch.load(save_path))

    # Final test evaluation
    test_dataset = EmotionDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)

    model.eval()
    all_test_preds = []
    all_test_labels = []

    with torch.no_grad():
        for sequences, labels in test_loader:
            sequences, labels = sequences.to(device), labels.to(device)
            outputs = model(sequences)
            _, predicted = torch.max(outputs, 1)

            all_test_preds.extend(predicted.cpu().numpy())
            all_test_labels.extend(labels.cpu().numpy())

    test_accuracy = sum(p == l for p, l in zip(all_test_preds, all_test_labels)) / len(all_test_preds)

    print(f"\nFINAL FIXED MODEL RESULTS:")
    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(all_test_labels, all_test_preds, target_names=emotions, zero_division=0, digits=3))

    return model, all_test_preds

In [24]:
model, preds = train_bilstm(X_train, X_val, X_test, y_train, y_val, y_test, emotions, device)

TRAINING FIXED BILSTM MODEL
Model architecture:
  Hidden size: 128
  Num layers: 1
  Dropout: 0.2
  Batch size: 64
Class weights: [1.2055652  2.6902857  2.6902857  0.39750084 2.5879192  1.6167582
 0.5245358 ]
Learning rate: 0.001
Weight decay: 1e-05
Using label smoothing: 0.1


Epoch 1: 100%|██████████| 663/663 [00:02<00:00, 232.38it/s]


Epoch  1: Train Loss: 1.7205, Train Acc: 41.99%, Val Loss: 1.5564, Val Acc: 55.45%, LR: 9.93e-04
           Predicting 7/7 different emotions
Saved best model with Val Acc: 55.45% on epoch 1


Epoch 2: 100%|██████████| 663/663 [00:02<00:00, 250.72it/s]


Epoch  2: Train Loss: 1.4819, Train Acc: 55.42%, Val Loss: 1.5063, Val Acc: 57.15%, LR: 9.95e-04
           Predicting 7/7 different emotions
Saved best model with Val Acc: 57.15% on epoch 2


...

Epoch 27: Train Loss: 0.7380, Train Acc: 95.23%, Val Loss: 1.6311, Val Acc: 63.16%, LR: 1.48e-04
           Predicting 7/7 different emotions


Epoch 28: 100%|██████████| 663/663 [00:02<00:00, 265.07it/s]


Epoch 28: Train Loss: 0.7286, Train Acc: 95.89%, Val Loss: 1.6545, Val Acc: 62.60%, LR: 8.40e-05
           Predicting 7/7 different emotions


Epoch 29: 100%|██████████| 663/663 [00:02<00:00, 265.63it/s]


Epoch 29: Train Loss: 0.7233, Train Acc: 96.37%, Val Loss: 1.6714, Val Acc: 61.97%, LR: 3.69e-05
           Predicting 7/7 different emotions
Early stopping at epoch 29

FINAL FIXED MODEL RESULTS:
Test Accuracy: 0.4344

Classification Report:
              precision    recall  f1-score   support

       anger      0.470     0.520     0.494       150
     disgust      0.100     0.068     0.081        44
        fear      0.256     0.244     0.250        41
         joy      0.746     0.404     0.524       225
     sadness      0.522     0.104     0.174       115
    surprise      0.367     0.136     0.198        81
     neutral      0.381     0.744     0.504       258

    accuracy                          0.434       914
   macro avg      0.406     0.317     0.318       914
weighted avg      0.483     0.434     0.407       914



In [27]:
def test_model_on_external_data(model, word2vec_model, device, emotions_names, text_column, label_column, filepath="../../data/transcript_spell_checked.csv"):
    """
    Test the trained model on external data
    """

    # Load and prepare external data
    df_external = pd.read_csv(filepath)
    df_external, _ = prepare_data(df_external, text_column, label_column)
    df_external["lemmatized_text"] = df_external["lemmatized_text"].apply(lambda tokens: " ".join(tokens))
    df_external["pos_tags"] = df_external[text_column].apply(lambda x: get_pos_tags(x, nlp))
    X_external, y_external = prepare_rnn_data(df_external, "lemmatized_text", word2vec_model, "ekman_emotion", additional_columns = ["pos_tags"])
    X_external = np.array(X_external)
    y_external = np.array(y_external)
    external_dataset = EmotionDataset(X_external, y_external)
    external_loader = DataLoader(external_dataset, batch_size=64, shuffle=False)
    print(f"External dataset size: {len(df_external)} samples")
    # Evaluate on external data
    model.eval()
    all_external_preds = []
    all_external_labels = []
    with torch.no_grad():
        for sequences, labels in external_loader:
            sequences, labels = sequences.to(device), labels.to(device)
            outputs = model(sequences)
            _, predicted = torch.max(outputs, 1)

            all_external_preds.extend(predicted.cpu().numpy())
            all_external_labels.extend(labels.cpu().numpy())
    external_accuracy = sum(p == l for p, l in zip(all_external_preds, all_external_labels)) / len(all_external_preds)
    print(f"\nEXTERNAL DATASET RESULTS:")
    print(f"External Data Accuracy: {external_accuracy:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(all_external_labels, all_external_preds, target_names=emotions_names, zero_division=0, digits=3))

In [28]:
test_model_on_external_data(model, word2vec_model, device, emotions, "Translation", "Emotion_core", filepath="../../data/kinga.csv")

Detected dataset type: transcript
Starting batch preprocessing...
✓ Text cleaning completed
✓ All NLP processing completed
Preparing RNN data from 791 samples...
Embedding dimension: 300
Max sequence length: 50


Converting to embeddings: 100%|██████████| 791/791 [00:00<00:00, 22588.25it/s]

Processing additional column: pos_tags
  Found 16 unique values in pos_tags
✓ Final shape: (791, 50, 301)
✓ OOV rate: 27.23% (1711/6283)
✓ Data type: float64
External dataset size: 791 samples

EXTERNAL DATASET RESULTS:
External Data Accuracy: 0.5171

Classification Report:
              precision    recall  f1-score   support

       anger      0.177     0.314     0.227        35
     disgust      0.400     0.222     0.286         9
        fear      0.833     0.086     0.156        58
         joy      0.465     0.484     0.474       153
     sadness      0.550     0.196     0.289        56
    surprise      0.232     0.284     0.255        67
     neutral      0.628     0.695     0.660       413

    accuracy                          0.517       791
   macro avg      0.469     0.326     0.335       791
weighted avg      0.550     0.517     0.503       791

